# 06c — Adaptation, extraction, and quantization — Practice

Read the lesson and run the example. Implement each TODO, then run its checks. A NotImplementedError is expected until you write your answer. Try for 20–30 minutes before opening a hint; open the solution only after recording an attempt. Copy this notebook into work/ to keep your own version.

**Prerequisites:** earlier lessons in the course order. **Environment:** Install requirements/core.txt.

[Stage guide](../stages/06.md) · [Course map](../PLAN.md) · [Project](../../projects/stage06/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Validate structured model outputs
- Explain and calculate low-rank updates
- Measure quantization error


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Lesson

Pretraining learns broad patterns; supervised fine-tuning adjusts weights on task examples. Full fine-tuning updates all parameters. LoRA learns low-rank matrices A and B and adds a scaled BA update to a frozen weight matrix. This reduces trainable parameters but does not remove the base model’s inference cost. PEFT implements parameter-efficient approaches. QLoRA combines adapters with a quantized frozen base and requires compatible kernels/hardware.

Structured extraction requires a schema, numeric units, evidence spans, and failure handling. JSON syntax alone does not prove that a value is supported by the source. Validate both structure and semantics; keep an unknown result when evidence is absent.

Quantization maps weights to a smaller representation. FP32/FP16/BF16 differ in precision and range; INT8/INT4 use integer codes and scaling metadata. Weight storage reduction does not guarantee proportional speedup because kernels, activation memory, and KV cache also matter. Measure quality on held-out examples after adaptation or quantization.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
import numpy as np
from pydantic import BaseModel, Field
print("A 1000×1000 matrix has", 1000*1000, "parameters")
print("Rank-8 adapters have", 8*(1000+1000), "parameters")


## Exercise 1: Validated extraction

Parse JSON with company (nonempty str) and revenue (finite float >=0), forbid extra keys. Return a validated model_dump dictionary.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def validate_extraction(text):
    raise NotImplementedError


In [ ]:
assert validate_extraction('{"company":"Example","revenue":100000000000}')["revenue"] == 1e11
from pydantic import ValidationError
expect_error(ValidationError, lambda: validate_extraction('{"company":"","revenue":-1}'))
print("Exercise 1: checks passed")


<details><summary>Hint — open after trying</summary>

Use model_validate_json.

</details>


## Exercise 2: LoRA update

For W(out,in), A(rank,in), B(out,rank), return W+(alpha/rank)*(B@A).

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def lora_weight(w, a, b, alpha):
    raise NotImplementedError


In [ ]:
w = np.zeros((3, 4)); a = np.ones((2, 4)); b = np.ones((3, 2))
assert np.allclose(lora_weight(w, a, b, 2), np.full((3, 4), 2))
print("Exercise 2: checks passed")


<details><summary>Hint — open after trying</summary>

Write down shapes before multiplying.

</details>


## Exercise 3: Symmetric INT8 simulation

Return int8 codes, scale, and dequantized weights using max(abs(w))/127; all-zero arrays use scale=1.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def quantize(w):
    raise NotImplementedError


In [ ]:
codes, scale, restored = quantize([-.7, .1, 1.])
assert codes.dtype == np.int8
assert np.max(np.abs(restored-[-.7, .1, 1.])) <= scale/2+1e-12
assert np.allclose(quantize([0, 0])[2], 0)
print("Exercise 3: checks passed")


<details><summary>Hint — open after trying</summary>

Store the scale along with integer codes.

</details>


## Independent transfer

Run the pretrained text-classification and PEFT lab in projects/stage06/pretrained.py after selecting/downloading a model. Compare frozen baseline versus fine-tuning on validation examples; record revision, license, cost, and test metrics. Audit extraction against source spans.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Explain without code

1. Does valid JSON imply truth?

2. What does LoRA change?


## Reading and review

- [Primary reference 1](https://huggingface.co/docs/peft/en/conceptual_guides/lora)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
